# Notebook 04 - Modélisation

**Projet** : Classification des Astéroïdes Potentiellement Dangereux (PHAs)
**Objectif** : Entraîner 4 modèles avec 3 stratégies de déséquilibre (12 configurations) via Validation Croisée Stratifiée (5 splits) sur les données préparées.

L'ensemble du code d'entraînement est intégré ici pour une transparence totale.

## 1. Imports et Configuration

In [1]:
import pandas as pd
import numpy as np
import time
import joblib
from pathlib import Path
from IPython.display import display

# Preprocessing et Pipeline
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Modèles
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

# Évaluation
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score, average_precision_score, roc_auc_score
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TARGET = "is_potentially_hazardous"

# Dossiers
PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

## 2. Chargement des données de la Phase 2

In [2]:
# On charge le train set généré en Phase 2 (le test set est réservé pour le notebook 06)
train_df = pd.read_csv(DATA_DIR / "train.csv")

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

print(f"Train set : {X_train.shape[0]:,} lignes × {X_train.shape[1]} colonnes")
print("Distribution des classes :")
print(y_train.value_counts(normalize=True) * 100)

# Chargement du preprocessor entraîné en Phase 2
preprocessor = joblib.load(MODELS_DIR / "preprocessor.joblib")
print("Preprocessor chargé avec succès.")

Train set : 12,000 lignes × 22 colonnes
Distribution des classes :
is_potentially_hazardous
0    90.241667
1     9.758333
Name: proportion, dtype: float64
Preprocessor chargé avec succès.


## 3. Définition des Modèles et Stratégies

Nous respectons les recommandations de la Phase 3 : 4 familles de modèles (Logistic Regression, Decision Tree, XGBoost, MLP) et 3 stratégies (Baseline avec class_weight, Oversampling avec SMOTE, Undersampling avec RandomUnderSampler).

In [3]:
# 1. Définition des Modèles de base
models = {
    "LogisticRegression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "DecisionTree": DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE),
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=4, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1),
    "MLPClassifier": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=200, early_stopping=True, random_state=RANDOM_STATE)
}

# 2. Définition des Stratégies
# Attention : XGBoost et MLP gèrent les class_weight un peu différemment, mais on simplifie ici.
# XGBoost a scale_pos_weight qu'on optimisera au tuning. Pour la baseline XGB/MLP on entraîne tel quel.
strategies = {
    "Baseline": None,
    "SMOTE": SMOTE(random_state=RANDOM_STATE),
    "RandomUnderSampler": RandomUnderSampler(random_state=RANDOM_STATE)
}

## 4. Validation Croisée (12 Configurations)

In [4]:
results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("Lancement de la validation croisée (cela peut prendre quelques minutes)...\n")

for model_name, model in models.items():
    for strat_name, sampler in strategies.items():
        print(f"Entraînement : {model_name} + {strat_name}...")
        
        # Copie du modèle pour ajuster les poids si c'est la baseline
        from sklearn.base import clone
        clf = clone(model)
        if strat_name == "Baseline" and hasattr(clf, "class_weight"):
            clf.set_params(class_weight="balanced")
            
        # Construction du pipeline (imblearn pipeline pour éviter le leakage)
        steps = [("preprocessor", preprocessor)]
        if sampler is not None:
            steps.append(("sampler", sampler))
        steps.append(("model", clf))
        
        pipeline = ImbPipeline(steps)
        
        # Stockage des scores des 5 folds
        f1_scores, pr_auc_scores, rec_scores, prec_scores = [], [], [], []
        
        for train_idx, val_idx in cv.split(X_train, y_train):
            X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
            y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
            
            pipeline.fit(X_fold_train, y_fold_train)
            
            # Pour PR-AUC, on a besoin des probabilités (ou de decision_function)
            if hasattr(pipeline, "predict_proba"):
                y_prob = pipeline.predict_proba(X_fold_val)[:, 1]
            else:
                y_prob = pipeline.decision_function(X_fold_val)
                
            y_pred = pipeline.predict(X_fold_val)
            
            f1_scores.append(f1_score(y_fold_val, y_pred))
            rec_scores.append(recall_score(y_fold_val, y_pred))
            prec_scores.append(precision_score(y_fold_val, y_pred))
            pr_auc_scores.append(average_precision_score(y_fold_val, y_prob))
            
        results.append({
            "Modèle": model_name,
            "Stratégie": strat_name,
            "F1 (mean)": np.mean(f1_scores), "F1 (std)": np.std(f1_scores),
            "Recall (mean)": np.mean(rec_scores),
            "Precision (mean)": np.mean(prec_scores),
            "PR-AUC (mean)": np.mean(pr_auc_scores)
        })

df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by=["F1 (mean)", "PR-AUC (mean)"], ascending=[False, False]).reset_index(drop=True)
df_results.to_csv(MODELS_DIR / "modeling_results.csv", index=False)
print("\nTerminé !")

Lancement de la validation croisée (cela peut prendre quelques minutes)...

Entraînement : LogisticRegression + Baseline...
Entraînement : LogisticRegression + SMOTE...
Entraînement : LogisticRegression + RandomUnderSampler...
Entraînement : DecisionTree + Baseline...
Entraînement : DecisionTree + SMOTE...
Entraînement : DecisionTree + RandomUnderSampler...
Entraînement : XGBoost + Baseline...
Entraînement : XGBoost + SMOTE...
Entraînement : XGBoost + RandomUnderSampler...
Entraînement : MLPClassifier + Baseline...
Entraînement : MLPClassifier + SMOTE...
Entraînement : MLPClassifier + RandomUnderSampler...

Terminé !


## 5. Tableau Comparatif

On trie selon le F1-score moyen pour sélectionner la meilleure combinaison pour le tuning.

In [5]:
# Mise en forme du tableau
formatted_results = df_results.copy()
for col in ["F1 (mean)", "F1 (std)", "Recall (mean)", "Precision (mean)", "PR-AUC (mean)"]:
    formatted_results[col] = formatted_results[col].map("{:.4f}".format)

formatted_results["F1 ± σ"] = formatted_results["F1 (mean)"] + " ± " + formatted_results["F1 (std)"]
display(formatted_results[["Modèle", "Stratégie", "F1 ± σ", "Recall (mean)", "Precision (mean)", "PR-AUC (mean)"]])

best_model = df_results.iloc[0]["Modèle"]
best_strat = df_results.iloc[0]["Stratégie"]
print(f"\nLe meilleur couple est : {best_model} avec {best_strat} (F1 = {df_results.iloc[0]['F1 (mean)']:.4f})")
print("C'est ce modèle qui sera optimisé dans le notebook 05_tuning.")

,Modèle,Stratégie,F1 ± σ,Recall (mean),Precision (mean),PR-AUC (mean)
0,XGBoost,Baseline,0.9876 ± 0.0034,0.9855,0.9898,0.9978
1,XGBoost,SMOTE,0.9868 ± 0.0031,0.9889,0.9847,0.9985
2,DecisionTree,Baseline,0.9804 ± 0.0073,0.9829,0.9781,0.9791
3,XGBoost,RandomUnderSampler,0.9782 ± 0.0033,0.9974,0.9598,0.9954
4,DecisionTree,SMOTE,0.9773 ± 0.0036,0.9752,0.9794,0.9693
5,DecisionTree,RandomUnderSampler,0.9604 ± 0.0107,0.9906,0.9323,0.9265
6,MLPClassifier,SMOTE,0.9464 ± 0.0061,0.9573,0.9363,0.9896
7,MLPClassifier,Baseline,0.9375 ± 0.0146,0.9283,0.9473,0.9876
8,MLPClassifier,RandomUnderSampler,0.8030 ± 0.0304,0.9778,0.6826,0.9440
9,LogisticRegression,SMOTE,0.7983 ± 0.0172,0.9590,0.6839,0.9327



Le meilleur couple est : XGBoost avec Baseline (F1 = 0.9876)
C'est ce modèle qui sera optimisé dans le notebook 05_tuning.
